In [44]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [45]:
pip install joblib

Note: you may need to restart the kernel to use updated packages.


In [46]:
# Import necessary libraries
import pandas as pd
import numpy as np
import joblib

In [47]:
import sklearn
print(sklearn.__version__) 

1.7.2


In [48]:
# Import necessary modules from scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline 
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV


In [49]:
# Load the dataset
df = pd.read_csv("C:\\Users\\NEETHU ANTONY A\\OneDrive\\Desktop\\Projects\\house_price_prediction\\ml_model\\dataset\\houses.csv")

In [50]:
# Display the first few rows of the dataset
df.head

<bound method NDFrame.head of      bedrooms  bathrooms  sqft_living  sqft_lot  floors  waterfront  view  \
0         3.0       1.00       1180.0    5650.0     1.0         0.0   0.0   
1         3.0       2.25       2570.0    7242.0     2.0         0.0   0.0   
2         2.0       1.00        770.0   10000.0     1.0         0.0   0.0   
3         4.0       3.00       1960.0    5000.0     1.0         0.0   0.0   
4         3.0       2.00       1680.0    8080.0     1.0         0.0   0.0   
..        ...        ...          ...       ...     ...         ...   ...   
995       4.0       2.50       1860.0    6325.0     2.0         0.0   0.0   
996       2.0       2.75       1590.0   20917.0     1.5         0.0   0.0   
997       2.0       1.00        850.0    2340.0     1.0         0.0   0.0   
998       2.0       1.00       1030.0    4188.0     1.0         0.0   0.0   
999       NaN        NaN          NaN       NaN     NaN         NaN   NaN   

     condition  grade  sqft_above  sqft_basem

In [51]:
# Check for missing values
df.isnull().sum()

bedrooms         1
bathrooms        1
sqft_living      1
sqft_lot         1
floors           1
waterfront       1
view             1
condition        1
grade            1
sqft_above       1
sqft_basement    1
yr_built         1
yr_renovated     1
zipcode          1
lat              1
long             1
sqft_living15    1
price            1
dtype: int64

In [52]:
# Drop rows with missing values
df=df.dropna()

In [53]:
df

,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,price
0,3.0,1.00,1180.0,5650.0,1.0,0.0,0.0,3.0,7.0,1180.0,0.0,1955.0,0.0,98178.0,47.5112,-122.257,1340.0,22.190
1,3.0,2.25,2570.0,7242.0,2.0,0.0,0.0,3.0,7.0,2170.0,400.0,1951.0,1991.0,98125.0,47.7210,-122.319,1690.0,53.800
2,2.0,1.00,770.0,10000.0,1.0,0.0,0.0,3.0,6.0,770.0,0.0,1933.0,0.0,98028.0,47.7379,-122.233,2720.0,18.000
3,4.0,3.00,1960.0,5000.0,1.0,0.0,0.0,5.0,7.0,1050.0,910.0,1965.0,0.0,98136.0,47.5208,-122.393,1360.0,60.400
4,3.0,2.00,1680.0,8080.0,1.0,0.0,0.0,3.0,8.0,1680.0,0.0,1987.0,0.0,98074.0,47.6168,-122.045,1800.0,51.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,2.0,1.00,740.0,6460.0,1.0,0.0,0.0,3.0,6.0,740.0,0.0,1953.0,0.0,98146.0,47.5077,-122.344,1170.0,17.850
995,4.0,2.50,1860.0,6325.0,2.0,0.0,0.0,4.0,7.0,1860.0,0.0,1991.0,0.0,98038.0,47.3492,-122.030,1860.0,29.100
996,2.0,2.75,1590.0,20917.0,1.5,0.0,0.0,3.0,5.0,1590.0,0.0,1920.0,0.0,98001.0,47.2786,-122.250,1310.0,19.995
997,2.0,1.00,850.0,2340.0,1.0,0.0,0.0,3.0,7.0,850.0,0.0,1922.0,0.0,98105.0,47.6707,-122.328,1300.0,55.350


In [54]:
df.isnull().sum()

bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          0
lat              0
long             0
sqft_living15    0
price            0
dtype: int64

In [55]:
# Separate features and target variable
X = df.drop("price", axis=1)
y = df["price"]


In [56]:
# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns

In [57]:
# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numerical_cols)])

In [58]:
#dict or list of dictionaries with parameters names (string) as keys and lists of parameter settings to try as values
param_grid = {
    "model__hidden_layer_sizes": [(64,32), (128,64), (100,)],
    "model__activation": ["relu", "tanh"],
    "model__learning_rate": ["constant", "adaptive"],
    "model__max_iter": [1000, 2000],
}

In [59]:
# Create an MLPRegressor model
model = MLPRegressor(
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)
# Create a pipeline that combines the preprocessor and the model
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])


In [60]:
#Apply GridSearchCV to find the best hyperparameters for the MLPRegressor
grid_search = GridSearchCV(
    #the pipeline that includes both preprocessing and the MLPRegressor model
    estimator=pipeline,    
    #the hyperparameters to search over
    param_grid=param_grid,  
    #use 5-fold cross-validation to evaluate the performance of each hyperparameter combination
    cv=5,   
    #use R-squared as the evaluation metric for regression
    scoring="r2",   
    #use all available CPU cores for parallel processing
    n_jobs=-1,  
    #this will print the progress of the grid search
    verbose=2   
)


In [61]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

# Train the model
grid_search.fit(X_train, y_train)

#Find best model
print("Best Hyperparameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

best_model = grid_search.best_estimator_

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Hyperparameters: {'model__activation': 'relu', 'model__hidden_layer_sizes': (100,), 'model__learning_rate': 'constant', 'model__max_iter': 1000}
Best CV score: 0.7562109205214249


In [62]:
y_pred = best_model.predict(X_test)

print("R2 Score:", r2_score(y_test, y_pred))
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
print("Root Mean Squared Error:", np.sqrt(mean_squared_error(y_test, y_pred)))

R2 Score: 0.7129146222016054
Mean Absolute Error: 10.43618212254532
Mean Squared Error: 258.10043710105134
Root Mean Squared Error: 16.06550457038469


In [63]:
import os
os.makedirs("models", exist_ok=True)

joblib.dump(best_model, "models/best_model.pkl")


['models/best_model.pkl']